# Exact Discrete Spectral MFPT vs Reset Probability r

This notebook plots the exact mean first passage time (MFPT) to consensus for the voter model on a complete graph using the discrete spectral expression:

$$\langle T(n_0)\rangle = \frac{\tilde{S}^{(1-r)}(n_0)}{1-r\tilde{S}^{(1-r)}(n_0)}$$

with high-precision arithmetic via `VoterResetting.exact_mfpt_discrete_spectral`.

Notes:
- The function is defined for $0 \le r < 1$.
- We still show the endpoint $r=1$ on the x-axis, but values there are set to `NaN` for plotting clarity.

In [ ]:
using Plots
using Printf

candidates = [abspath(pwd()), abspath(joinpath(pwd(), "..")), abspath(joinpath(pwd(), "..", ".."))]
project_root = nothing
for p in candidates
    if isfile(joinpath(p, "src", "VoterResetting.jl"))
        global project_root = p
        break
    end
end
project_root === nothing && error("Could not locate project root containing src/VoterResetting.jl from pwd=$(pwd())")

include(joinpath(project_root, "src", "VoterResetting.jl"))
const VR = VoterResetting

default(size = (980, 540), linewidth = 2)

In [ ]:
# Parameters
N = 120
m0_values = [-0.6, -0.3, 0.0, 0.3, 0.6]
nr = 220
precision_bits = 512

r_grid = collect(range(0.0, 1.0, length = nr))
r_eps = 1e-14

function m0_to_n0(N::Int, m0::Real)
    n0_raw = N * (1 + m0) / 2
    n0 = round(Int, n0_raw)
    return clamp(n0, 1, N - 1)
end

function mfpt_curve_for_m0(N::Int, m0::Real, r_values; precision_bits::Int = 512)
    n0 = m0_to_n0(N, m0)
    m0_eff = 2n0 / N - 1

    curve = Vector{Float64}(undef, length(r_values))
    for (idx, r) in enumerate(r_values)
        if r >= 1.0
            curve[idx] = NaN
            continue
        end

        r_eval = min(r, 1.0 - r_eps)
        val = VR.exact_mfpt_discrete_spectral(
            N, n0, r_eval;
            precision_bits = precision_bits,
            return_bigfloat = true,
        )
        curve[idx] = Float64(val)
    end

    return (n0 = n0, m0_eff = m0_eff, r = r_values, mfpt = curve)
end

In [ ]:
curves = [mfpt_curve_for_m0(N, m0, r_grid; precision_bits = precision_bits) for m0 in m0_values]

p = plot(
    xlabel = "r",
    ylabel = "Exact MFPT",
    title = "Discrete spectral MFPT vs r (N=$(N), high precision)",
    legend = :topright,
    yscale = :log10,
)

for (m0_req, c) in zip(m0_values, curves)
    label_txt = @sprintf("m0 req=%.2f, n0=%d, m0 eff=%.3f", m0_req, c.n0, c.m0_eff)
    plot!(p, c.r, c.mfpt, label = label_txt)
end

vline!(p, [1.0], label = "r = 1 (excluded in formula)", linestyle = :dash, color = :black, alpha = 0.35)
p

In [ ]:
println("Run summary:")
@printf("  N = %d\n", N)
@printf("  points in r grid = %d\n", length(r_grid))
@printf("  BigFloat precision_bits = %d\n", precision_bits)
println("  m0 requested = " * string(m0_values))
println("  m0 effective (due to integer n0) = " * string([c.m0_eff for c in curves]))